<a href="https://colab.research.google.com/github/imgeG/CYT-300-Group-1/blob/M2/CYT300M2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classifying Phishing Emails with Malicious URLs using AI

## Milestone 2 — Dataset Collection

In [ ]:
import pandas as pd
import numpy as np
import os
import re

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "naserabdullahalam/phishing-email-dataset"
)

print("Dataset path:", path)

Using Colab cache for faster access to the 'phishing-email-dataset' dataset.
Dataset path: /kaggle/input/phishing-email-dataset


In [ ]:
print("Files in dataset folder:")

for file in os.listdir(path):
    print(file)

Files in dataset folder:
SpamAssasin.csv
Nazario.csv
Nigerian_Fraud.csv
CEAS_08.csv
Enron.csv
Ling.csv
phishing_email.csv


In [ ]:
ceas_df = pd.read_csv(os.path.join(path, "CEAS_08.csv"))
enron_df = pd.read_csv(os.path.join(path, "Enron.csv"))
ling_df = pd.read_csv(os.path.join(path, "Ling.csv"))
nazario_df = pd.read_csv(os.path.join(path, "Nazario.csv"))
nigerian_df = pd.read_csv(os.path.join(path, "Nigerian_Fraud.csv"))
spamassassin_df = pd.read_csv(os.path.join(path, "SpamAssasin.csv"))

print("All six datasets loaded successfully.")

All six datasets loaded successfully.


In [ ]:
datasets = {
    "CEAS_08": ceas_df,
    "Enron": enron_df,
    "Ling": ling_df,
    "Nazario": nazario_df,
    "Nigerian_Fraud": nigerian_df,
    "SpamAssasin": spamassassin_df
}

In [ ]:
for name, data in datasets.items():
    print(name)
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())

CEAS_08
Shape: (39154, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
Enron
Shape: (29767, 3)
Columns: ['subject', 'body', 'label']
Ling
Shape: (2859, 3)
Columns: ['subject', 'body', 'label']
Nazario
Shape: (1565, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
Nigerian_Fraud
Shape: (3332, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
SpamAssasin
Shape: (5809, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']


In [ ]:
for name, data in datasets.items():
    print(name)
    print(data["label"].value_counts(dropna=False).sort_index())

CEAS_08
label
0    17312
1    21842
Name: count, dtype: int64
Enron
label
0    15791
1    13976
Name: count, dtype: int64
Ling
label
0    2401
1     458
Name: count, dtype: int64
Nazario
label
1    1565
Name: count, dtype: int64
Nigerian_Fraud
label
1    3332
Name: count, dtype: int64
SpamAssasin
label
0    4091
1    1718
Name: count, dtype: int64


In [ ]:
for name, data in datasets.items():
    print("\n" + "=" * 70)
    print(name)

    print("Rows:", len(data))
    print("Missing values:")
    print(data.isna().sum())

    print("\nDuplicate rows:", data.duplicated().sum())


CEAS_08
Rows: 39154
Missing values:
sender        0
receiver    462
date          0
subject      28
body          0
label         0
urls          0
dtype: int64

Duplicate rows: 0

Enron
Rows: 29767
Missing values:
subject    198
body         0
label        0
dtype: int64

Duplicate rows: 0

Ling
Rows: 2859
Missing values:
subject    62
body        0
label       0
dtype: int64

Duplicate rows: 0

Nazario
Rows: 1565
Missing values:
sender       0
receiver    96
date         1
subject      4
body         0
urls         0
label        0
dtype: int64

Duplicate rows: 0

Nigerian_Fraud
Rows: 3332
Missing values:
sender       331
receiver    1324
date         482
subject       39
body           0
urls           0
label          0
dtype: int64

Duplicate rows: 0

SpamAssasin
Rows: 5809
Missing values:
sender        0
receiver    210
date          0
subject      16
body          1
label         0
urls          0
dtype: int64

Duplicate rows: 0


In [ ]:
def standardize_dataset(df, source_name):
    df = df.copy()

    # Add missing columns where necessary
    required_columns = [
        "sender",
        "receiver",
        "date",
        "subject",
        "body",
        "urls",
        "label"
    ]

    for column in required_columns:
        if column not in df.columns:
            df[column] = pd.NA

    # Keep only the standardized columns
    df = df[required_columns]

    # Add source name
    df.insert(0, "source", source_name)

    return df

In [ ]:
ceas_std = standardize_dataset(ceas_df, "CEAS_08")
enron_std = standardize_dataset(enron_df, "Enron")
ling_std = standardize_dataset(ling_df, "Ling")
nazario_std = standardize_dataset(nazario_df, "Nazario")
nigerian_std = standardize_dataset(nigerian_df, "Nigerian_Fraud")
spamassassin_std = standardize_dataset(spamassassin_df, "SpamAssasin")

In [ ]:
df_combined = pd.concat(
    [
        ceas_std,
        enron_std,
        ling_std,
        nazario_std,
        nigerian_std,
        spamassassin_std
    ],
    ignore_index=True
)

print("Combined dataset shape:", df_combined.shape)
print("\nColumns:")
print(df_combined.columns.tolist())

Combined dataset shape: (82486, 8)

Columns:
['source', 'sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']


In [ ]:
def create_text_combined(row):
    parts = []

    if pd.notna(row["sender"]):
        parts.append("Sender: " + str(row["sender"]))

    if pd.notna(row["date"]):
        parts.append("Date: " + str(row["date"]))

    if pd.notna(row["subject"]):
        parts.append("Subject: " + str(row["subject"]))

    if pd.notna(row["body"]):
        parts.append("Body: " + str(row["body"]))

    return " ".join(parts)


df_combined["text_combined"] = df_combined.apply(
    create_text_combined,
    axis=1
)

print("text_combined created successfully.")
print("New shape:", df_combined.shape)

text_combined created successfully.
New shape: (82486, 9)


In [ ]:
print("Shape:", df_combined.shape)

print("\nColumns:")
print(df_combined.columns.tolist())

print("\nFirst 3 emails:")
print(
    df_combined[
        ["source", "subject", "label", "text_combined"]
    ].head(3).to_string(index=False)
)

Shape: (82486, 9)

Columns:
['source', 'sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label', 'text_combined']

First 3 emails:
 source                   subject  label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [ ]:
print("Missing text_combined:", df_combined["text_combined"].isna().sum())

empty_text = (
    df_combined["text_combined"]
    .fillna("")
    .str.strip()
    .eq("")
)

print("Empty text_combined:", empty_text.sum())

Missing text_combined: 0
Empty text_combined: 0


In [ ]:
df_combined.to_csv("combined_email_dataset.csv", index=False)

print("Saved successfully!")
print("Rows:", len(df_combined))
print("Columns:", len(df_combined.columns))

Saved successfully!
Rows: 82486
Columns: 9


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ndarvind/phiusiil-phishing-url-dataset")

print("Path to dataset files:", path)

100%|██████████| 14.7M/14.7M [00:00<00:00, 151MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ndarvind/phiusiil-phishing-url-dataset/versions/1


In [ ]:
import os

print(os.listdir(path))

['PhiUSIIL_Phishing_URL_Dataset.csv']


In [ ]:
import pandas as pd
import os

phi_file = os.path.join(path, "PhiUSIIL_Phishing_URL_Dataset.csv")

phiusiil_df = pd.read_csv(phi_file)

print("Shape:", phiusiil_df.shape)
print("\nColumns:")
print(phiusiil_df.columns.tolist())

Shape: (235795, 55)

Columns:
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [ ]:
phiusiil_df.duplicated().sum()

np.int64(0)

In [ ]:
phiusiil_df["URL"].duplicated().sum()

np.int64(425)

In [ ]:
print(phiusiil_df["label"].value_counts())

label
1    134850
0    100945
Name: count, dtype: int64


In [ ]:
import pandas as pd

phishtank_url = "http://data.phishtank.com/data/online-valid.csv"

phishtank_df = pd.read_csv(phishtank_url)

print("Shape:", phishtank_df.shape)
print("\nColumns:")
print(phishtank_df.columns.tolist())

Shape: (76728, 8)

Columns:
['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified', 'verification_time', 'online', 'target']


In [ ]:
print("Verified:")
print(phishtank_df["verified"].value_counts())

print("\nOnline:")
print(phishtank_df["online"].value_counts())

Verified:
verified
yes    76728
Name: count, dtype: int64

Online:
online
yes    76728
Name: count, dtype: int64


In [ ]:
print("Duplicate rows:", phishtank_df.duplicated().sum())
print("Duplicate URLs:", phishtank_df["url"].duplicated().sum())

Duplicate rows: 0
Duplicate URLs: 3


In [ ]:
phishtank_df.to_csv("phishtank_phishing_urls.csv", index=False)

print("Saved successfully!")
print("Rows:", len(phishtank_df))
print("Columns:", len(phishtank_df.columns))


Saved successfully!
Rows: 76728
Columns: 8


In [ ]:
dataset_summary = pd.DataFrame({
    "Dataset": [
        "Combined Email Dataset",
        "PhiUSIIL",
        "PhishTank"
    ],
    "Type": [
        "Email",
        "URL",
        "URL"
    ],
    "Records": [
        len(df_combined),
        len(phiusiil_df),
        len(phishtank_df)
    ],
    "Columns": [
        len(df_combined.columns),
        len(phiusiil_df.columns),
        len(phishtank_df.columns)
    ]
})

display(dataset_summary)

,Dataset,Type,Records,Columns
0,Combined Email Dataset,Email,82486,9
1,PhiUSIIL,URL,235795,55
2,PhishTank,URL,76728,8


In [ ]:
print("Combined Email Dataset")
print(df_combined["label"].value_counts(dropna=False))

print("\nPhiUSIIL")
print(phiusiil_df["label"].value_counts(dropna=False))

print("\nPhishTank")
print("Verified:")
print(phishtank_df["verified"].value_counts(dropna=False))


Combined Email Dataset
label
1    42891
0    39595
Name: count, dtype: int64

PhiUSIIL
label
1    134850
0    100945
Name: count, dtype: int64

PhishTank
Verified:
verified
yes    76728
Name: count, dtype: int64
